# 02. kNN auf IMDb mit Wort-Bigrammen

**Leitfrage:** Wie viele der 100 Validierungsreviews klassifiziert unser bekanntes kNN richtig, wenn Wortpaare hinzukommen?

**Arbeitsauftrag:** Führe den [Lauf mit Einzelwörtern](#baseline) aus und notiere die Trefferzahl. Ergänze im [zweiten Lauf](#bigrams) die Bigramme, prüfe die Spalten für `not bad` und `not good` und vergleiche die beiden Ergebnisse.

In [ ]:
%pip install scikit-learn==1.9.1

<a id='baseline'></a>

## 1. Den bekannten Lauf mit Einzelwörtern wiederholen

Wie im kNN-Notebook lesen wir jede Review aus einer UTF-8-Datei und das Label `pos` oder `neg` aus ihrem Ordnernamen. Die Nummer im Dateinamen bestimmt die Reihenfolge. [Kursdaten und Herkunft](data/imdb/README.md)

In [ ]:
from pathlib import Path

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KNeighborsClassifier

data_dir = Path('data/imdb')
train_folder = data_dir / 'train'

In [ ]:
def load_reviews(folder: Path) -> tuple[list[str], list[str]]:
    paths = sorted(folder.glob('*/*.txt'), key=lambda path: path.name)
    texts = []
    labels = []
    for path in paths:
        texts.append(path.read_text(encoding='utf-8'))
        labels.append(path.parent.name)
    return texts, labels

In [ ]:
train_texts, train_labels = load_reviews(train_folder)
validation_texts, validation_labels = load_reviews(data_dir / 'validation')
print('Training:', len(train_texts), '| Validation:', len(validation_texts))

Lerne den Wortschatz mit `fit_transform` auf dem Training und verwende ihn mit `transform` für die Validierung. Der Klassifikator stimmt mit drei Nachbarn und Kosinusdistanz ab. [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) · [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

In [ ]:
vectorizer = CountVectorizer()
train_vectors = vectorizer.fit_transform(train_texts)
validation_vectors = vectorizer.transform(validation_texts)

In [ ]:
imdb_model = KNeighborsClassifier(n_neighbors=3, metric='cosine')
imdb_model.fit(train_vectors, train_labels)
predictions = imdb_model.predict(validation_vectors)

In [ ]:
correct = sum(predictions == validation_labels)
print('Validation correct:', correct, '/', len(validation_labels))
print(f'Accuracy: {correct / len(validation_labels):.0%}')

<a id='bigrams'></a>

## 2. Denselben Lauf um Bigramme erweitern

Setze beim neuen `CountVectorizer` den Parameter `ngram_range=(1, 2)`. Die Grenzen 1 und 2 stehen für Einzelwörter und Wortpaare. Lerne die erweiterten Wortspalten wieder am Training und führe die folgenden Zellen aus.

In [ ]:
# TODO: Include both single words and adjacent word pairs.
vectorizer = CountVectorizer()
train_vectors = vectorizer.fit_transform(train_texts)
validation_vectors = vectorizer.transform(validation_texts)

Die folgende Zelle zeigt nur die Wortspalten für `not bad` und `not good`. Sage für Review A und B die beiden Zählwerte voraus.

In [ ]:
review_a = 'The film was good, not bad.'
review_b = 'The film was bad, not good.'
features = vectorizer.get_feature_names_out()
columns = (features == 'not bad') | (features == 'not good')
print(features[columns])
print(vectorizer.transform([review_a, review_b])[:, columns].toarray())

In [ ]:
imdb_model = KNeighborsClassifier(n_neighbors=3, metric='cosine')
imdb_model.fit(train_vectors, train_labels)
predictions = imdb_model.predict(validation_vectors)

In [ ]:
correct = sum(predictions == validation_labels)
print('Validation correct:', correct, '/', len(validation_labels))
print(f'Accuracy: {correct / len(validation_labels):.0%}')

Notiere die Trefferzahl für jeden Lauf. Wie groß ist der Unterschied auf denselben 100 Validierungsreviews? Welche Einstellung hast du geändert, welche sind für den Vergleich gleich geblieben?